# Pix2Struct DocVQA-base — DIMER OCR-free document question answering tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-docvqa-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-docvqa-pipeline/blob/main/tutorials/pix2struct_docvqa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fpix2struct--docvqa--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-docvqa-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2210.03347-b31b1b.svg)](https://arxiv.org/abs/2210.03347)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** OCR-free document question answering — one page image plus one natural-language question → one short answer string — using the pinned `google/pix2struct-docvqa-base` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/pix2struct_docvqa_pipeline/pipeline.py` at revision `71f3e9b08c05`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `63f6b3de436e39f75c7a486881a9c2c14a7f4e89` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the Pix2Struct image-encoder/text-decoder (a ViT-style encoder over variable-resolution 16×16 patches and a 12-layer text decoder, 282M parameters, pretrained by parsing masked web screenshots into HTML and fine-tuned on DocVQA) reads the **question rendered as a text header above the page** — the Pix2Struct convention for visual question answering — scales the composite to fill at most 2048 patches, and generates the answer text token by token. Decoding is greedy (`do_sample=False`) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, a non-empty question up to 256 characters, the token budget), a fixed output contract, an offline header font (Pillow's bundled Aileron replaces the Hub font the upstream processor would otherwise download), and the `anls`, `exact_match`, `validate_inputs` and `evaluation_report` helpers. The default sample is an invoice-style form rendered in code with five authored questions and accepted answers, so ANLS and exact-match are demonstration (plumbing) evidence for one page, not a DocVQA benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, render a synthetic form with authored question/answer pairs (or upload your own page and write your own questions) and validate it into an input manifest, choose a token budget, run the supported task, read the answers correctly (generated text, no score, a `truncated` flag), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `anls` and `exact_match` only when accepted answers exist and `not-measurable` otherwise, and export the answers, the annotated page and provenance.

**This notebook does not demonstrate:** PDF or multi-page documents (one page image per call), OCR output or answer localisation (the model returns text, not the region it read), questions that need arithmetic or reasoning across pages, batch throughput, sampling or beam search, evaluation on the DocVQA benchmark (registration-gated and not bundled; only authored questions on a rendered page are scored here), and any training. The model was fine-tuned on scanned business documents in English; photographs, handwriting, non-Latin scripts and long free-text answers are outside what this notebook measures, and a fluent wrong answer carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.3 s to load and 2.3–3.0 s per question on the 850×1100 rendered form in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 1.13 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; what normalised Levenshtein similarity (ANLS) measures; that a confident answer is not a correct one.
- **Data:** the default sample is a deterministic 850×1100 invoice-style form rendered in code with Pillow's bundled font (header, supplier, five labelled fields, a four-row line-item table, totals, a payment-terms line) with five authored questions and their accepted answers, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar) of a **single document page**, any colour mode, sides between 16 and 4096 px, plus your own questions typed into the form field. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-docvqa-base` snapshot (~1133 MB in total) at revision `63f6b3de436e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-docvqa-pipeline',
    'repository_revision': '71f3e9b08c050fea8ef8f06fd0d1cd559d6075cf',
    'embedded_module': 'src/pix2struct_docvqa_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_docvqa_pipeline/pipeline.py'],
    'module_sha256': '7f369485a5bf28424df9c4cd53cede2883e89f092894f6fa4f5401e6fc519089',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_docvqa_pipeline/` @ `71f3e9b08c05`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/pix2struct_docvqa_pipeline/pipeline.py`

In [ ]:
"""OCR-free document question answering with the pinned ``google/pix2struct-docvqa-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The question is rendered as a
text header on top of the page (the Pix2Struct VQA input convention) with Pillow's bundled font, so no
font is fetched from the Hub at inference time.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image, ImageFont

MODEL_ID = "google/pix2struct-docvqa-base"
MODEL_REVISION = "63f6b3de436e39f75c7a486881a9c2c14a7f4e89"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-docvqa-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Generation ceilings. DocVQA answers are short spans (the checkpoint's text_config max_length is 20);
# the default leaves room for a long address or title, the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 128
DEFAULT_MAX_NEW_TOKENS = 32
DECODING = "greedy"
# Question ceiling. The question is rendered as a header line (wrapped at 80 characters by the
# processor) above the page; a very long question shrinks the page's share of the patch budget.
MAX_QUESTION_CHARS = 256
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget, so pixel count only guards memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# ANLS (DocVQA's official metric): a normalised Levenshtein similarity below this threshold scores 0.
ANLS_THRESHOLD = 0.5
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def header_font_bytes() -> bytes:
    """Pillow's bundled Aileron Regular (CC0) as TrueType bytes: the header font for the rendered question.

    The upstream image processor otherwise fetches ``ybelkada/fonts/Arial.TTF`` from the Hub at
    inference time — an unpinned, unlisted download of a proprietary font. The bundled subset covers
    the printable ASCII range, which is what a question is expected to use.
    """
    font = ImageFont.load_default(size=36)
    data = getattr(font, "font_bytes", None)
    if not data:
        raise RuntimeError("Pillow's bundled TrueType font is unavailable (FreeType support missing)")
    return bytes(data)


def normalize_answer(text: str) -> str:
    """DocVQA-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def _levenshtein(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def anls(prediction: str, golds: Sequence[str], *, threshold: float = ANLS_THRESHOLD) -> float:
    """Average Normalised Levenshtein Similarity for one question (Biten et al., ICDAR 2019).

    ``1 - lev(pred, gold) / max(len(pred), len(gold))`` over normalised strings, maximised over the
    accepted ``golds``; a similarity below ``threshold`` scores 0 so a near-miss is not rewarded.
    """
    if not golds:
        raise ValueError("golds must contain at least one accepted answer")
    pred = normalize_answer(prediction)
    best = 0.0
    for gold in golds:
        ref = normalize_answer(gold)
        longest = max(len(pred), len(ref))
        similarity = 1.0 if longest == 0 else 1.0 - _levenshtein(pred, ref) / longest
        best = max(best, similarity)
    return best if best >= threshold else 0.0


def exact_match(prediction: str, golds: Sequence[str]) -> bool:
    """Whether the normalised prediction equals any normalised accepted answer."""
    pred = normalize_answer(prediction)
    return any(pred == normalize_answer(gold) for gold in golds)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one page image as PIL.Image.Image (any mode, converted to RGB) plus one question string",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "the question is rendered as a black-on-white header (Pillow's bundled font, wrapped at 80 "
        "characters) above the page; the composite is scaled to fill at most MAX_PATCHES 16x16 patches "
        "(aspect ratio preserved), normalised per image, and flattened into patch tokens with row/column "
        "positions; the decoder generates the answer text"
    ),
    "output": "one answer string (the model's decoded text), no score",
}


def _check_inputs(image: Any, question: Any, max_new_tokens: Any) -> tuple[Image.Image, str, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``answer`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked_question = " ".join(question.split())
    if not checked_question:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked_question) > MAX_QUESTION_CHARS:
        raise ValueError(
            f"question has {len(checked_question)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}"
        )
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_question, max_new_tokens


def validate_inputs(
    image: Image.Image,
    questions: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every question is checked exactly as ``answer`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(questions, str) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of str")
    checked = [_check_inputs(image, question, max_new_tokens)[1] for question in questions]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one page image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "questions": checked,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    golds: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one sequence of accepted answers per result, in order) the report carries the
    mean ``anls`` and the ``exact_match`` rate over the questions plus one per-question entry, verdict
    ``sample-sanity``; without golds it is ``not-measurable`` and says what labelled data would make
    the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "document page image + question -> answer text (OCR-free)",
        "score_semantics": (
            "the answer is generated text and carries no score, probability or correctness signal; a "
            "fluent answer is not evidence that it is read from the page. Greedy decoding makes the "
            "output reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no accepted answers were supplied for the evaluated questions",
            "needs": (
                "question/answer pairs with accepted answers on pages from the deployment domain "
                "(DocVQA-style annotations) scored with ANLS; no such labelled set ships with this repository"
            ),
        }
    if len(golds) != len(results):
        raise ValueError(f"golds has {len(golds)} entries for {len(results)} results")
    per_question = []
    for result, accepted in zip(results, golds, strict=True):
        if isinstance(accepted, str) or not accepted:
            raise ValueError("each golds entry must be a non-empty sequence of accepted answers")
        prediction = str(result["answer"])
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": prediction,
                "golds": list(accepted),
                "anls": anls(prediction, accepted),
                "exact_match": exact_match(prediction, accepted),
            }
        )
    metrics = [
        {
            "id": "anls",
            "value": sum(entry["anls"] for entry in per_question) / len(per_question),
            "threshold": ANLS_THRESHOLD,
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; max over golds",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
        {
            "id": "exact_match",
            "value": sum(entry["exact_match"] for entry in per_question) / len(per_question),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial page whose content you rendered "
            "yourself; plumbing evidence, not a DocVQA benchmark"
        ),
        "needs": (
            "a labelled question/answer set on pages from the deployment domain (scans, forms, layouts) "
            "for any accuracy claim; the DocVQA benchmark itself is registration-gated and not bundled"
        ),
    }


@dataclass
class Pix2StructDocVQAPipeline:
    """``_runner(image, question, max_new_tokens)`` returns ``{"answer": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructDocVQAPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        font_bytes = header_font_bytes()
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if not getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is not the VQA variant (is_vqa=False); refusing")
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, question: str, max_new_tokens: int) -> dict[str, Any]:
            # The image processor is called directly: Pix2StructProcessor.__call__ drops the
            # font_bytes kwarg, and font_bytes is what replaces the default Hub font download
            # (see header_font_bytes). The VQA processor renders the question as the header.
            inputs = processor.image_processor(
                image, header_text=question, return_tensors="pt", font_bytes=font_bytes
            ).to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            # Encoder-decoder: the output holds only decoder tokens (decoder_start + answer + eos).
            answer_ids = generated[0]
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"answer": decoded, "new_tokens": int(answer_ids.shape[0]) - 1}

        return cls(runner, resolved_device, "float32", source)

    def answer(
        self,
        image: Image.Image,
        question: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Answer one question about one page image; ``answer`` is the decoded text, stripped."""
        rgb, checked_question, checked_tokens = _check_inputs(image, question, max_new_tokens)
        raw = self._runner(rgb, checked_question, checked_tokens)
        if not isinstance(raw, dict) or "answer" not in raw:
            raise RuntimeError("runner must return a dict with 'answer'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "answer": str(raw["answer"]).strip(),
            "question": checked_question,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `63f6b3de436e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructDocVQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-docvqa-base",
  "modelId": "google/pix2struct-docvqa-base",
  "revision": "63f6b3de436e39f75c7a486881a9c2c14a7f4e89",
  "files": [
    {
      "path": "README.md",
      "bytes": 4476,
      "sha256": "794175546e80948e4efef30e95ebe859fdd26d07bcda26f738a1c97feb1e912a"
    },
    {
      "path": "config.json",
      "bytes": 4892,
      "sha256": "8d39973772a4218b555e30daecabdd5ea11aa1345dd711ff7f88fa90750b464f"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "067f7f314d87fa56daa5bcfaf36fa0b33ceebf7b7d4fae6a1e51ab7af64ee0b5"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "c84e4eebc84171d6069533d9f0147ec7b4afd02ab78697cb5c30f9419ef7dc45"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 1133308924
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructDocVQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Render the synthetic form or optional BYOD

The default sample is **synthetic** and carries its own references: an invoice-style form — an `INVOICE` header, a supplier name and address, five labelled fields (invoice number, dates, customer, purchase order), a four-row line-item table, subtotal/VAT/total lines and a payment-terms sentence — is rendered with Pillow's bundled font at 850×1100, the same page the repository's smoke run used. Five questions are authored against it, each with the accepted answer(s) as written on the page; they are the references for the `anls` and `exact_match` sanity checks later. They are not a labelled dataset, so nothing here is a DocVQA measurement. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one page image and type your questions (one per line) — no accepted answers exist for them, so the evaluation report will be `not-measurable`.

The token budget is a **caller-owned request parameter**: `max_new_tokens` bounds the answer (`DEFAULT_MAX_NEW_TOKENS = 32` fits any field on this form; `MAX_NEW_TOKENS = 128` is the ceiling). Nothing is validated in this cell — the next section hands the image and the questions to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the page size and digest, the budget and the number of questions.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
byod_questions = 'What is the invoice number?\nWho is the customer?'  # @param {type:"string"}
max_new_tokens = 32  # @param {type:"integer"}


def synthetic_form(width=850, height=1100):
    """An invoice-style form rendered with Pillow's bundled font; returns page + [(question, accepted answers)]."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    body, bold, head = ImageFont.load_default(size=18), ImageFont.load_default(size=20), ImageFont.load_default(size=30)
    d.text((70, 60), 'INVOICE', fill='black', font=head)
    d.text((70, 110), 'Northwind Traders Ltd.', fill='black', font=bold)
    d.text((70, 136), '14 Harbour Road, Portsmouth PO1 3AX', fill=(40, 40, 40), font=body)
    fields = [('Invoice number:', 'NW-2026-0417'), ('Invoice date:', '12 March 2026'), ('Due date:', '11 April 2026'), ('Customer:', 'Blue Yonder Airlines'), ('Purchase order:', 'PO-88213')]
    y = 200
    for label, value in fields:
        d.text((70, y), label, fill='black', font=bold)
        d.text((300, y), value, fill='black', font=body)
        y += 32
    d.rectangle([70, 400, 780, 640], outline='black', width=2)
    cols = [70, 420, 540, 660, 780]
    rows = [('Description', 'Qty', 'Unit price', 'Amount'), ('Cargo pallets (standard)', '40', '$18.50', '$740.00'), ('Shrink wrap rolls', '12', '$9.25', '$111.00'), ('Handling fee', '1', '$65.00', '$65.00')]
    for r, row in enumerate(rows):
        yy = 400 + r * 48
        if r:
            d.line([(70, yy), (780, yy)], fill=(120, 120, 120), width=1)
        for c, cell in enumerate(row):
            d.text((cols[c] + 10, yy + 14), cell, fill='black', font=bold if r == 0 else body)
    for c in cols[1:-1]:
        d.line([(c, 400), (c, 640)], fill=(120, 120, 120), width=1)
    d.text((540, 670), 'Subtotal:', fill='black', font=bold)
    d.text((680, 670), '$916.00', fill='black', font=body)
    d.text((540, 700), 'VAT (20%):', fill='black', font=bold)
    d.text((680, 700), '$183.20', fill='black', font=body)
    d.text((540, 736), 'Total due:', fill='black', font=head)
    d.text((680, 736), '$1,099.20', fill='black', font=head)
    d.text((70, 900), 'Payment terms: 30 days from invoice date. Bank: Solent Mutual, sort code 40-11-22.', fill=(40, 40, 40), font=body)
    qa = [
        ('What is the invoice number?', ['NW-2026-0417']),
        ('Who is the customer?', ['Blue Yonder Airlines']),
        ('What is the total due?', ['$1,099.20', '1,099.20']),
        ('What is the due date?', ['11 April 2026']),
        ('How many cargo pallets were invoiced?', ['40']),
    ]
    return page, qa


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    questions = [line.strip() for line in byod_questions.splitlines() if line.strip()]
    golds = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic form: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image, qa = synthetic_form()
    questions, golds = [q for q, _ in qa], [g for _, g in qa]
    image_name = 'synthetic_invoice_850x1100.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'max_new_tokens': max_new_tokens, 'n_questions': len(questions), 'has_golds': golds is not None})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `answer` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, each question a non-empty string of at most `MAX_QUESTION_CHARS` characters (whitespace collapsed), and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the header-rendering preprocessing and the decoding rule), the input's observed mode and size, the checked questions, the budget and the verdict. The manifest is written to `outputs/pix2struct_docvqa_input_manifest.json`. To show what rejection looks like, the cell also validates a blank question and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB, the question is rendered above it, and the composite is scaled to the patch budget; nothing else is dropped or altered. The pipeline cannot tell whether the image is a document or whether the question is answerable from it: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}})
input_manifest = validate_inputs(image, questions, max_new_tokens=max_new_tokens, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, ['   '])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'blank-question-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/pix2struct_docvqa_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Answer the questions and read the output correctly

`answer` returns, per question, a dict with `answer` (the decoded text, stripped), the checked `question`, `image_size`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: the answer is generated text with no probability and no correctness signal, and a fluent answer is not evidence that it was read from the page. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection can change a token and therefore the rest of the answer, so GPU and CPU outputs need not match. Each call renders the question as a header and re-encodes the page, so cost is per question (about 2.3–3.0 s each on the reference CPU). As recorded in the model card, the repository's CPU smoke on this same form answered all five authored questions exactly — and answered `51759 7951` to "What is the invoice number?" on a blank 4096×4096 page: the model always produces an answer, whether or not one exists.

In [ ]:
import time

results, seconds = [], []
for question in questions:
    t0 = time.time()
    results.append(pipe.answer(image, question, max_new_tokens=max_new_tokens))
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'dtype': pipe.dtype, 'seconds_per_question': seconds, 'any_truncated': any(r['truncated'] for r in results)})
for result in results:
    print(f"Q: {result['question']}\n   A: {result['answer']!r}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})")
if any(r['truncated'] for r in results):
    print('A budget was exhausted: that answer is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No accuracy is reported by default: DocVQA-style accuracy needs labelled question/answer pairs on pages from the deployment domain, and this repository ships none (the DocVQA benchmark itself is registration-gated). The repository's metric helpers are `anls` — Average Normalised Levenshtein Similarity, the benchmark's official metric: `1 − edits / max(len)` over normalised strings, maximised over the accepted answers, scored 0 below the 0.5 threshold — and `exact_match` after the same normalisation (lower-case, punctuation removed, whitespace collapsed). When accepted answers are supplied the report carries the mean `anls`, the `exact_match` rate and one entry per question, with the verdict `sample-sanity`. On the synthetic path those answers are values **you rendered yourself**, so a perfect score proves only that the input contract, header rendering, forward pass and decoding round-trip. On BYOD no accepted answers exist, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/pix2struct_docvqa_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, golds, sample_kind=sample_kind)
with open('outputs/pix2struct_docvqa_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_question')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:12} {metric['value']:.3f}  ({metric['estimation']})")
for entry in report.get('per_question', []):
    print(f"  anls {entry['anls']:.2f}  exact {str(entry['exact_match']):5}  {entry['question']} -> {entry['prediction']!r} (accepted: {entry['golds']})")
if report['verdict'] == 'not-measurable':
    print('No accepted answers exist for these questions, so nothing is scored; read the answers against the page yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every result (question, answer, `new_tokens`, `truncated`, the budget), the evaluation report, the input manifest, the sample identity, digest and accepted answers, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The question/answer pairs are also written as CSV with explicit `image`, `question`, `answer`, `new_tokens`, `truncated` columns, and an annotated PNG shows the page with the questions and answers printed in a panel beneath it for visual inspection (the model returns no location, so nothing is drawn on the page itself) — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

panel_height = 30 + 26 * len(results)
annotated = Image.new('RGB', (image.width, image.height + panel_height), 'white')
annotated.paste(image.convert('RGB'), (0, 0))
draw = ImageDraw.Draw(annotated)
draw.line([(0, image.height + 1), (image.width, image.height + 1)], fill=(120, 120, 120), width=2)
panel_font = ImageFont.load_default(size=16)
for index, result in enumerate(results):
    draw.text((20, image.height + 12 + 26 * index), f"{result['question']}  ->  {result['answer']}", fill=(40, 90, 220), font=panel_font)
annotated.save('outputs/pix2struct_docvqa_annotated.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'questions': questions, 'accepted_answers': golds},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/pix2struct_docvqa_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/pix2struct_docvqa_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'question', 'answer', 'new_tokens', 'truncated'])
    for result in results:
        writer.writerow([image_name, result['question'], result['answer'], result['new_tokens'], result['truncated']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The answers are the text the model generates after reading a page with the question printed above it; nothing in the output scores that text, the model returns no location or evidence, and it answers every question — including one about a blank page — with equal fluency. On the synthetic form the `anls` and `exact_match` values in the evaluation report compare the answers with values you rendered yourself and the verdict is `sample-sanity`, which proves only that the input contract, header rendering, forward pass and decoding work (the repository's smoke run scored 5/5 exact on this page); they say nothing about scans, photographs, dense multi-column layouts, handwriting, non-Latin scripts, questions that need arithmetic or reasoning, or answers longer than a field, and a BYOD result is a single-page observation with the verdict `not-measurable`. **The model answers any question about any image** and stops only at end-of-sequence or the token budget: check `truncated`, and treat a plausible answer to an unanswerable question as the expected failure mode, not an exception. The pipeline provides no OCR, no answer localisation, no multi-page handling, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** ask a question the form cannot answer (`What is the delivery address?`) and see the model invent one; ask `What is the VAT amount?` and compare with `$183.20`; lower `max_new_tokens` to 2 and watch `truncated` turn true; enable `USE_BYOD` with a page you know, type your questions, then pass your own accepted answers to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/pix2struct-docvqa-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/pix2struct-docvqa-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/pix2struct-docvqa-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/pix2struct-docvqa-base
- Upstream code: https://github.com/google-research/pix2struct
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., 2022): https://arxiv.org/abs/2210.03347
- DocVQA: A Dataset for VQA on Document Images (Mathew, Karatzas, Jawahar, 2020): https://arxiv.org/abs/2007.00398
- Scene Text Visual Question Answering — the ANLS metric (Biten et al., 2019): https://arxiv.org/abs/1905.13648